In [1]:
import os

TITANIC_PATH = '/cxldata/datasets/project/titanic'

In [2]:
# load train & test data
import pandas as pd

def load_titanic_data(filename, titanic_path=TITANIC_PATH):
    csv_path = os.path.join(titanic_path, filename)
    return pd.read_csv(csv_path)

train_data = load_titanic_data("train.csv")
test_data = load_titanic_data("test.csv")


In [3]:
# Create preprocessing pipeline

from sklearn.base import BaseEstimator, TransformerMixin

class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names):
        self.attribute_names = attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.attribute_names]

In [4]:
# creating pipeline for numerical attributes

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
        ("select_numeric", 
         DataFrameSelector(["Age", "SibSp", "Parch", "Fare"])),
        ("imputer", SimpleImputer(strategy="median")),
    ])

num_pipeline.fit_transform(train_data)

array([[22.    ,  1.    ,  0.    ,  7.25  ],
       [38.    ,  1.    ,  0.    , 71.2833],
       [26.    ,  0.    ,  0.    ,  7.925 ],
       ...,
       [28.    ,  1.    ,  2.    , 23.45  ],
       [26.    ,  0.    ,  0.    , 30.    ],
       [32.    ,  0.    ,  0.    ,  7.75  ]])

In [5]:
# Create Imputer for String Categorical Columns
# regular SimpleImputer does not work in this case

class MostFrequentImputer (BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.most_frequent_ = pd.Series([X[c].value_counts().index[0] for c in X],
                                        index=X.columns)
        return self
    def transform(self, X, y=None):
        return X.fillna(self.most_frequent_)

In [6]:
# Build pipeline for categorical attributes

from sklearn.preprocessing import OneHotEncoder # convert categorical data into a numeric format

cat_pipeline = Pipeline([
        ("select_cat", DataFrameSelector(["Pclass", "Sex", "Embarked"])),
        ("imputer", MostFrequentImputer()),
        ("cat_encoder", OneHotEncoder(sparse=False)),
    ])

cat_pipeline.fit_transform(train_data)

array([[0., 0., 1., ..., 0., 0., 1.],
       [1., 0., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 0., 1.],
       ...,
       [0., 0., 1., ..., 0., 0., 1.],
       [1., 0., 0., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 1., 0.]])

In [7]:
# join both pipelines

from sklearn.pipeline import FeatureUnion
preprocess_pipeline = FeatureUnion(transformer_list=[
        ("num_pipeline", num_pipeline),
        ("cat_pipeline", cat_pipeline),
    ])

X_train = preprocess_pipeline.fit_transform(train_data)

y_train = train_data["Survived"]

In [8]:
# Train SVC classifier (Support Vector Classifier)

from sklearn.svm import SVC

svm_clf = SVC(gamma="auto", random_state=42)
svm_clf.fit(X_train, y_train)

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='auto', kernel='rbf',
    max_iter=-1, probability=False, random_state=42, shrinking=True, tol=0.001,
    verbose=False)

In [9]:
# Predict using test set

X_test = preprocess_pipeline.transform(test_data)
y_pred = svm_clf.predict(X_test)

In [10]:
# Evaluate SVC model

from sklearn.model_selection import cross_val_score

svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
svm_scores.mean()

0.7329588014981274

In [11]:
# Train Random FOrest Classifier

from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
forest_scores.mean()

0.8126466916354558

In [ ]:
# 1. Imports
import os
import pandas as pd # load and handle data
from sklearn.base import BaseEstimator, TransformerMixin
# BaseEstimator - Makes it “sklearn-compatible”, allows parameter handling
# TransformerMixin - Gives transformation behavior <fit_transform()>
# These two are not doing your logic—they are making your custom class plug into sklearn’s ecosystem properly
    
from sklearn.pipeline import Pipeline, FeatureUnion # build structured pre-processing
from sklearn.impute import SimpleImputer # fill missing values
from sklearn.preprocessing import OneHotEncoder # convert categorical data into a numeric format
# from sklearn.pipeline import FeatureUnion
from sklearn.svm import SVC # model
from sklearn.ensemble import RandomForestClassifier # model
from sklearn.model_selection import cross_val_score # evaluate model
# data => clean => transform => train => evaluate

# 2. Load Data
TITANIC_PATH = '/cxldata/datasets/project/titanic'

def load_titanic_data(filename, titanic_path=TITANIC_PATH):
    csv_path = os.path.join(titanic_path, filename)
    return pd.read_csv(csv_path)
train_data = load_titanic_data("train.csv")
test_data = load_titanic_data("test.csv")


# 3. Custom Transformer: DataFrameSelector (Create preprocessing pipeline)
class DataFrameSelector(BaseEstimator, TransformerMixin): # This class should behave like a proper sklearn transformer.
    def __init__(self, attribute_names):
        self.attribute_names = attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.attribute_names]
    
# Numerical Pipeline for numerical attributes
# PassengerId: just a unique identifier, no predictive meaning
# Pclass: Categorical (ordinal), not true numeric
num_pipeline = Pipeline([("select_numeric", DataFrameSelector(["Age", "SibSp", "Parch", "Fare"])), # excluded PassengerId and Pclass 
        ("imputer", SimpleImputer(strategy="median")), # median is robust to outliers
    ])
num_pipeline.fit_transform(train_data)

# 5. Custom Imputer for Categorical Data
# regular SimpleImputer does not work in this case
class MostFrequentImputer (BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.most_frequent_ = pd.Series(
            [X[c].value_counts().index[0] for c in X],
            index=X.columns)
        return self
    def transform(self, X, y=None):
        return X.fillna(self.most_frequent_) # replaces missing values (NaN) in a dataset with something you specify

# 6. Categorical Pipeline for categorical attributes
cat_pipeline = Pipeline([
        ("select_cat", DataFrameSelector(["Pclass", "Sex", "Embarked"])),
        ("imputer", MostFrequentImputer()),
        ("cat_encoder", OneHotEncoder(sparse=False)), # give a regular full array instead of a compressed sparse matrix
    ])

cat_pipeline.fit_transform(train_data)


# 7. Combine both pipelines
preprocess_pipeline = FeatureUnion(transformer_list=[
        ("num_pipeline", num_pipeline),
        ("cat_pipeline", cat_pipeline),
    ])

# 8. Prepare training data
X_train = preprocess_pipeline.fit_transform(train_data)
y_train = train_data["Survived"]


# 9. Train SVM model (SVC classifier (Support Vector Classifier))
# SVM Model : SVM chooses the line that separates the classes with the maximum margin
# can be used for classification & regression. Mostly used for classification
svm_clf = SVC(gamma="auto", random_state=42)
svm_clf.fit(X_train, y_train)

# 10. Predict on test data
X_test = preprocess_pipeline.transform(test_data)
y_pred = svm_clf.predict(X_test)


# 11. Evaluate SVC model
svm_scores = cross_val_score(svm_clf, X_train, y_train, cv=10)
svm_scores.mean()


# 12. Train Random Forest Classifier
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
forest_scores.mean()